# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(bool(os.path.exists(FINAL_DOC)))

True


#### Load Data

In [3]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

(15310889, 14)

In [4]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

In [5]:
df['lang'] = [lang(x) for x in tqdm(df['corpus'])]

  0%|          | 0/15310889 [00:00<?, ?it/s]

## Convert values to numeric tags

In [11]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker', 
    # 'dyad', 
    'lang',
    'groups'
    #'x_speaker_turn'
]

In [12]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}
    
    if isinstance(col,list):
        for c  in col:
            df[c+'_'] = [conversion[v] for v in tqdm(df[c])]
    
    else:
        df[col+'_'] = [conversion[v] for v in tqdm(df[col])]


  0%|          | 0/15310889 [00:00<?, ?it/s]

  0%|          | 0/15310889 [00:00<?, ?it/s]

### Other data preparation processes

In [ ]:
df.head()

## LME Regression: Predicting CE

In [10]:
import statsmodels.formula.api as smf

In [ ]:
# df = df.loc[~df['null']]

### Resid model

In [13]:
df['g'] = 0

In [14]:
##########################################
## A set of resids to show ∇H / t
##########################################

model = "Hxy ~ nx + ny + (1|groups_) + (2|lang_)"
##########################################

In [15]:
start = dt.now()
# md = smf.mixedlm(model, data=df, groups=df['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['g'])
mdf = md.fit()
print('completed in:', dt.now()-start)

completed in: 0:00:52.577984


In [16]:
# mdf.summary()
mdf.bse

Intercept               NaN
nx             1.636653e-05
ny             1.639067e-05
1 | groups_    1.228041e-09
2 | lang_      4.817517e-05
Group Var               NaN
dtype: float64

In [17]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

,coefs,stat,p,"CI[.025, .975]"
Intercept,-1.05e-01,nan,nan,"[nan, nan]"
nx,2.79e-01,1.71e+04,0.e+00,"[2.79e-01, 2.79e-01]"
ny,-2.53e-02,-1.54e+03,0.e+00,"[-2.53e-02, -2.53e-02]"
1 | groups_,-5.72e-09,-4.66e+00,3.2e-06,"[-8.13e-09, -3.31e-09]"
2 | lang_,-1.42e-02,-2.95e+02,0.e+00,"[-1.43e-02, -1.41e-02]"
Group Var,1.e+00,nan,nan,"[nan, nan]"


#### Adding residuals into the mix

Because the resid values are really useful for visualizations in particular.

In [18]:
df['resid'] = mdf.resid

#### Tidying the data

Making sure there aren't any erroneous `\t` elements in there.

In [19]:
# # cleaning
# for col in list(df):
#     print(col)
#     df[col] = [it.replace('\t', '') if isinstance(it,str) else it for it in tqdm(df[col])]

#### Save residuals

In [19]:
# df.to_csv(FINAL_DOC, index=False, encoding='utf-8', sep='\t')
df.to_parquet(FINAL_DOC)

#### check that residuals file is working . . .

In [20]:
df_ = pd.read_parquet(FINAL_DOC)

In [21]:
del df_